# Package

In [1]:
# ============================================================
# 1) Core Python & Paths
# ============================================================
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from dataclasses import dataclass
from typing import Callable, Dict, Any, Optional, List, Iterable

from dateutil.relativedelta import relativedelta

# Project root (notebook dans /notebooks)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 2) Scientific Stack (NumPy / Pandas)
# ============================================================
import numpy as np
import pandas as pd


# ============================================================
# 3) Machine Learning & Models
# ============================================================
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import (
    GridSearchCV,
    ParameterGrid,
    ParameterSampler
)

from lightgbm import LGBMRegressor


# ============================================================
# 4) Forecasting (Nixtla MLForecast)
# ============================================================
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals


# ============================================================
# 5) Data & Project Utilities
# ============================================================
from utils import load_wide_from_feast, build_unrate_exog_dataset
from experiment_utils import to_wide_from_oos, make_mae_dm_pivot


# ============================================================
# 6) Statistical Tests
# ============================================================
from scipy.stats import ttest_rel
from statsmodels.stats.contingency_tables import mcnemar


# ============================================================
# 7) Visualization
# ============================================================
from utilsforecast.plotting import plot_series
from IPython.display import IFrame, display


# ============================================================
# 8) Experiment Tracking (MLflow)
# ============================================================
import mlflow
from mlflow.tracking import MlflowClient

# Importation des données

In [2]:
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO",
    "M2SL","OILPRICEX","RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1960-01-01"
END   = "2025-08-01"

In [3]:
# ============================================================
# 1) WIDE dataset (df) depuis Feast
# ============================================================

LAG = 12  

df = load_wide_from_feast(
    "stationary_value:value",
    series_ids,
    start=START,
    end=END
)

# ============================================================
# 2) Convert WIDE -> LONG (obligatoire pour build_unrate_exog_dataset)
# ============================================================
df_stationary = (
    df.reset_index()
      .melt(id_vars="date", var_name="series_id", value_name="value")
)

# ============================================================
# 3) Build target + exog + MLForecast format
# ============================================================
df_model, ts_lr, exog_cols = build_unrate_exog_dataset(df_stationary)

# ============================================================
# 4) ✅ TOUTES les exog laggées de 12 mois (aucune contemporaine)
#    - on crée BUSLOANS_lag12, CPIAUCSL_lag12, ...
#    - on supprime BUSLOANS, CPIAUCSL, ... (contemporaines)
# ============================================================
ts_lr = ts_lr.sort_values(["unique_id", "ds"]).copy()

exog_cols_lag12 = []
for c in exog_cols:
    new_c = f"{c}_lag{LAG}"
    ts_lr[new_c] = ts_lr.groupby("unique_id")[c].shift(LAG)
    exog_cols_lag12.append(new_c)

# supprimer les exog contemporaines
ts_lr = ts_lr.drop(columns=exog_cols)

# mettre à jour la liste exog utilisée par MLForecast
exog_cols = exog_cols_lag12

# drop lignes où les lag12 n'existent pas (12 premiers mois)
ts_lr = ts_lr.dropna(subset=["y"] + exog_cols).reset_index(drop=True)

# ============================================================
# 5) Prints / checks
# ============================================================
print("df (wide) shape:", df.shape)
print("df_stationary (long) shape:", df_stationary.shape)
print("df_model shape:", df_model.shape)
print("ts_lr shape (after lag12):", ts_lr.shape)
print("Exog cols (lag12):", exog_cols)

print("\nPreview:")
print(ts_lr[["unique_id", "ds", "y"] + exog_cols[:5]].head(15))

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df (wide) shape: (788, 11)
df_stationary (long) shape: (8668, 3)
df_model shape: (788, 12)
ts_lr shape (after lag12): (776, 13)
Exog cols (lag12): ['BUSLOANS_lag12', 'CPIAUCSL_lag12', 'DPCERA3M086SBEA_lag12', 'INDPRO_lag12', 'M2SL_lag12', 'OILPRICEX_lag12', 'RPI_lag12', 'SP500_lag12', 'TB3MS_lag12', 'USREC_lag12']

Preview:
   unique_id         ds    y  BUSLOANS_lag12  CPIAUCSL_lag12  \
0     UNRATE 1961-01-01  1.4        0.011578       -0.006156   
1     UNRATE 1961-02-01  2.1        0.011905       -0.003767   
2     UNRATE 1961-03-01  1.5       -0.008356       -0.005455   
3     UNRATE 1961-04-01  1.8       -0.009098        0.005090   
4     UNRATE 1961-05-01  2.0       -0.000359        0.003383   
5     UNRATE 1961-06-01  1.5        0.014620        0.006777   
6     UNRATE 1961-07-01  1.5       -0.000611       -0.005433   
7     UNRATE 1961-08-01  1.0       -0.016888       -0.004074   

d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\ML Experiment\utils.py:171: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


# Model settings

In [ ]:
# -----------------------------
# Helpers temps
# -----------------------------
def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

def _slice_cv_block(ts, cutoff_start, n_windows, h):
    cutoff_end = cutoff_start + relativedelta(months=n_windows - 1)
    ds_end = cutoff_end + relativedelta(months=h)
    return ts[ts["ds"] <= ds_end].copy(), cutoff_end, ds_end

In [5]:
# -----------------------------
# Spec modèle (tu ajoutes juste ici)
# -----------------------------
@dataclass
class ModelSpec:
    name: str                         # "LR", "RIDGE", "LGBM", etc.
    build_mlf: Callable[[str, Dict[str, Any]], MLForecast]  # (freq, params)->MLForecast
    pred_col: str                     # colonne de forecast dans cv (ex "LR")
    tunable: bool = False
    param_space: Optional[Dict[str, Iterable[Any]]] = None
    search: str = "random"            # "random"|"grid"
    n_iter: int = 50                  # si random
    tune_cv_windows: int = 6
    tune_every_months: int = 36
    use_conformal_in_tune: bool = False
    fixed_params: Optional[Dict[str, Any]] = None


In [ ]:
# -----------------------------
# Tuner générique (pour tous les modèles)
# -----------------------------
def _tune_on_train(
    ts_train: pd.DataFrame,
    *,
    spec: ModelSpec,
    freq: str,
    h: int,
    levels: List[int],
    seed: int,
    pi_windows_cap: int,
    min_train_n: Optional[int] = None,
) -> tuple[Optional[Dict[str, Any]], float]:

    if min_train_n is not None and len(ts_train) < int(min_train_n):
        return None, float("nan")

    if not spec.tunable:
        return (spec.fixed_params or {}), float("nan")

    if not spec.param_space:
        raise ValueError(f"{spec.name}: tunable=True mais param_space=None")

    # conformal au tuning (souvent OFF)
    pi_tune = (
        PredictionIntervals(h=h, n_windows=min(int(spec.tune_cv_windows), int(pi_windows_cap)),
                            method="conformal_distribution")
        if spec.use_conformal_in_tune else None
    )

    # générateur d'essais
    if spec.search == "grid":
        sampler = ParameterGrid(spec.param_space)
    else:
        sampler = ParameterSampler(spec.param_space, n_iter=int(spec.n_iter), random_state=int(seed))

    best_params = None
    best_score = np.inf

    for params in sampler:
        params = dict(params)

        mlf = spec.build_mlf(freq, params)
        cv = mlf.cross_validation(
            df=ts_train,
            h=h,
            step_size=1,
            n_windows=int(spec.tune_cv_windows),
            prediction_intervals=pi_tune,
            level=list(levels) if pi_tune is not None else None,
            fitted=False,
            static_features=[],
            dropna=True,
        )
        score = mean_absolute_error(cv["y"], cv[spec.pred_col])

        if score < best_score:
            best_score = float(score)
            best_params = params

    return best_params, float(best_score)

In [7]:
# -----------------------------
# Runner multi-modèles : tu ajoutes juste un spec dans la liste
# -----------------------------
def run_backtesting_generic(
    ts: pd.DataFrame,
    *,
    model_specs: List[ModelSpec],
    freq: str,
    h: int,
    exp_start,
    exp_end,
    step_size: int,
    pi_windows: int,
    levels: List[int],
    seed: int = 0,
    min_train_n: Optional[int] = None,
) -> tuple[pd.DataFrame, Dict[str, Any]]:

    # run modèle par modèle puis merge
    bkts = []
    metas = {}

    for spec in model_specs:
        bkt_m, meta_m = backtest_one_model_tune_blocks(
            ts,
            spec=spec,
            freq=freq,
            h=h,
            exp_start=exp_start,
            exp_end=exp_end,
            step_size=step_size,
            pi_windows=pi_windows,
            levels=levels,
            seed=seed,
            min_train_n=min_train_n,
        )
        metas[spec.name] = meta_m
        if len(bkt_m):
            bkts.append(bkt_m)

    if not bkts:
        return pd.DataFrame(), {"error": "aucun modèle n’a produit de backtest", "metas": metas}

    # merge wide sur clés
    keys = ["unique_id", "ds", "cutoff", "y"]
    bkt_all = bkts[0].copy()

    for b in bkts[1:]:
        # éviter collisions si certains champs internes existent
        keep_cols = [c for c in b.columns if c not in bkt_all.columns or c in keys]
        bkt_all = bkt_all.merge(b[keep_cols], on=keys, how="outer")

    bkt_all = bkt_all.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

    # ✅ 1 ligne par ds : dernier cutoff
    bkt_final = (
        bkt_all.sort_values(["unique_id", "ds", "cutoff"])
               .groupby(["unique_id", "ds"], as_index=False)
               .tail(1)
               .reset_index(drop=True)
    )

    meta = {"metas": metas}
    return bkt_final, meta

In [ ]:
# -----------------------------
# Backtesting générique avec retrain+tuning par blocs
# -> renvoie bkt wide pour 1 modèle (colonne spec.pred_col + PI)
# -----------------------------
def backtest_one_model_tune_blocks(
    ts: pd.DataFrame,
    *,
    spec: ModelSpec,
    freq: str,
    h: int,
    exp_start,
    exp_end,
    step_size: int,
    pi_windows: int,
    levels: List[int],
    seed: int = 0,
    min_train_n: Optional[int] = None,
) -> tuple[pd.DataFrame, Dict[str, Any]]:

    ts = ts.copy()
    ts["ds"] = (
        pd.to_datetime(ts["ds"], errors="coerce")
          .dt.to_period("M")
          .dt.to_timestamp(how="start")
          .dt.normalize()
    )
    if ts["ds"].isna().any():
        bad = ts[ts["ds"].isna()].head()
        raise ValueError(f"Dates 'ds' invalides après parsing. Exemples:\n{bad}")

    exp_start = _ensure_ms(exp_start)
    exp_end   = _ensure_ms(exp_end)

    cutoff_start_all = exp_start - relativedelta(months=h)
    cutoff_end_all   = exp_end   - relativedelta(months=h)
    total_partitions = _n_windows_monthly(cutoff_start_all, cutoff_end_all)

    # anti-fuite
    ts = ts[ts["ds"] <= exp_end].copy()

    # conformal au backtest (ici ON)
    pi = PredictionIntervals(h=h, n_windows=int(pi_windows), method="conformal_distribution")

    # blocs de retrain/tuning
    blocks = []
    remaining = int(total_partitions)
    cur = cutoff_start_all
    while remaining > 0:
        n_win = min(int(spec.tune_every_months), remaining)
        blocks.append((cur, n_win))
        cur = cur + relativedelta(months=n_win)
        remaining -= n_win

    all_bkts = []
    params_history = []
    tune_history = []

    for block_idx, (cutoff_start_blk, n_windows_blk) in enumerate(blocks, start=1):
        ts_blk, _, _ = _slice_cv_block(ts, cutoff_start_blk, int(n_windows_blk), h)

        ts_train_for_tune = ts[ts["ds"] <= cutoff_start_blk].copy()
        if min_train_n is not None and len(ts_train_for_tune) < int(min_train_n):
            continue

        best_params, tune_mae = _tune_on_train(
            ts_train_for_tune,
            spec=spec,
            freq=freq,
            h=h,
            levels=levels,
            seed=seed,
            pi_windows_cap=int(pi_windows),
            min_train_n=min_train_n,
        )
        if best_params is None:
            continue

        params_history.append({
            "model": spec.name,
            "block": block_idx,
            "cutoff_start": cutoff_start_blk,
            "n_windows": int(n_windows_blk),
            **best_params,
        })
        tune_history.append({
            "model": spec.name,
            "block": block_idx,
            "cutoff_start": cutoff_start_blk,
            "tune_mae": float(tune_mae),
        })

        mlf_blk = spec.build_mlf(freq, best_params)

        bkt_blk = mlf_blk.cross_validation(
            df=ts_blk,
            h=h,
            step_size=int(step_size),
            n_windows=int(n_windows_blk),
            prediction_intervals=pi,
            level=list(levels),
            fitted=True,
            static_features=[],
            dropna=True,
        )
        bkt_blk[f"{spec.name}_tune_block"] = block_idx
        bkt_blk[f"{spec.name}_tune_mae"] = float(tune_mae)

        all_bkts.append(bkt_blk)

    if not all_bkts:
        return pd.DataFrame(), {"error": f"{spec.name}: aucun bloc produit"}

    bkt = pd.concat(all_bkts, ignore_index=True)

    # filtre exp exact
    bkt = bkt[(bkt["ds"] >= exp_start) & (bkt["ds"] <= exp_end)].copy()
    bkt = bkt.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

    meta = dict(
        model=spec.name,
        h=int(h),
        step_size=int(step_size),
        exp_start=exp_start,
        exp_end=exp_end,
        cutoff_start=cutoff_start_all,
        cutoff_end=cutoff_end_all,
        partitions=int(total_partitions),
        pi_windows=int(pi_windows),
        tune_every_months=int(spec.tune_every_months),
        tune_cv_windows=int(spec.tune_cv_windows),
        tunable=bool(spec.tunable),
        search=spec.search,
        n_iter=int(spec.n_iter),
        use_conformal_in_tune=bool(spec.use_conformal_in_tune),
        param_space=spec.param_space,
        params_history=params_history,
        tune_history=tune_history,
    )
    return bkt, meta


In [9]:
# ============================================================
# Builders modèles
# ============================================================

def build_lr(freq, params):
    return MLForecast(models={"LR": LinearRegression()}, freq=freq, lags=[], date_features=[])

def build_ridge(freq, params):
    alpha = params.get("alpha", 1.0)
    return MLForecast(models={"RIDGE": Ridge(alpha=alpha)}, freq=freq, lags=[], date_features=[])

LGBM_BASE = dict(random_state=0, n_jobs=-1, verbosity=-1, objective="regression", metric="mae")

def build_lgbm(freq, params):
    p = dict(LGBM_BASE)
    p.update(params)
    return MLForecast(models={"LGBM": LGBMRegressor(**p)}, freq=freq, lags=[], date_features=[])

# ============================================================
# Model Specs (LR + RIDGE + LGBM)
# ============================================================

MODEL_SPECS = [
    ModelSpec(
        name="LR",
        build_mlf=build_lr,
        pred_col="LR",
        tunable=False,
        fixed_params={},
    ),
    ModelSpec(
        name="RIDGE",
        build_mlf=build_ridge,
        pred_col="RIDGE",
        tunable=True,
        param_space={"alpha": np.logspace(-4, 4, 30)},  # ✅ corrigé
        search="grid",
        tune_every_months=36,
    ),
    ModelSpec(
        name="LGBM",
        build_mlf=build_lgbm,
        pred_col="LGBM",
        tunable=True,
        param_space={
            "subsample":        [0.05, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0],
            "colsample_bytree": [.2, .3, .4, .5, .6, .7, 1.0],
            "num_leaves":       [2, 3, 4, 5, 8, 10, 20, 40, 70, 100],
            "n_estimators":     [5, 10, 20, 30, 40, 50, 75, 100],
            "max_depth":        [1, 2, 3, 5, 8, 15, -1],
            "reg_alpha":        [0, .1, 1, 2, 7, 10, 50, 100],
            "reg_lambda":       [0, .1, 1, 10, 20, 50, 100],
            "min_child_samples":[5, 10, 15],
            "min_split_gain":   [0.0, 0.01, 0.05],
        },  # ✅ accolade corrigée
        search="random",
        n_iter=12,
        tune_every_months=36,
    ),
]

In [10]:
# 2) Run LR + RIDGE (avec retrain/tuning par blocs pour Ridge)
H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]
FREQ = "MS"
EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")
SEED = 42

bkt_models_final, meta_models = run_backtesting_generic(
    ts=ts_lr,
    model_specs=MODEL_SPECS,   # ← ta liste avec LR + RIDGE + LGBM
    freq=FREQ,
    h=H,
    exp_start=EXP_START,
    exp_end=EXP_END,
    step_size=STEP_SIZE,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
    seed=SEED,
    min_train_n=36,
)

print("✅ rows:", len(bkt_models_final))
print("✅ columns (head):", bkt_models_final.columns.tolist()[:25])

# vérifier blocs Ridge
print("✅ Ridge blocks:", len(meta_models["metas"]["RIDGE"].get("params_history", [])))

# vérifier blocs LGBM
print("✅ LGBM blocks:", len(meta_models["metas"]["LGBM"].get("params_history", [])))

bkt_models_final.head()

✅ rows: 428
✅ columns (head): ['unique_id', 'ds', 'cutoff', 'y', 'LR', 'LR-lo-95', 'LR-hi-95', 'LR_tune_block', 'LR_tune_mae', 'RIDGE', 'RIDGE-lo-95', 'RIDGE-hi-95', 'RIDGE_tune_block', 'RIDGE_tune_mae', 'LGBM', 'LGBM-lo-95', 'LGBM-hi-95', 'LGBM_tune_block', 'LGBM_tune_mae']
✅ Ridge blocks: 12
✅ LGBM blocks: 12


,unique_id,ds,cutoff,y,LR,LR-lo-95,LR-hi-95,LR_tune_block,LR_tune_mae,RIDGE,RIDGE-lo-95,RIDGE-hi-95,RIDGE_tune_block,RIDGE_tune_mae,LGBM,LGBM-lo-95,LGBM-hi-95,LGBM_tune_block,LGBM_tune_mae
0,UNRATE,1990-01-01,1989-12-01,0.0,-0.219232,-0.847277,0.408814,1,NaN,-0.161217,-0.667560,0.345126,1,0.486142,-0.009485,-0.854084,0.835113,1,0.495416
1,UNRATE,1990-02-01,1990-01-01,0.1,-0.496136,-0.599595,-0.392678,1,NaN,-0.359640,-0.605153,-0.114128,1,0.486142,0.233700,0.031543,0.435856,1,0.495416
2,UNRATE,1990-03-01,1990-02-01,0.2,-0.221596,-0.463243,0.020051,1,NaN,-0.249758,-0.418144,-0.081371,1,0.486142,0.106741,-0.448775,0.662258,1,0.495416
3,UNRATE,1990-04-01,1990-03-01,0.2,-0.152979,-0.408678,0.102721,1,NaN,-0.307614,-0.715236,0.100009,1,0.486142,-0.068158,-0.543226,0.406911,1,0.495416
4,UNRATE,1990-05-01,1990-04-01,0.2,-0.127317,-0.720993,0.466358,1,NaN,-0.408729,-0.860543,0.043086,1,0.486142,0.125031,-0.646020,0.896082,1,0.495416


# Transform Backtesting

In [14]:
bkt_score = bkt_models_final.copy()
bkt_score["ds"] = pd.to_datetime(bkt_score["ds"], errors="coerce")

# enlever timezone si jamais
if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):
    bkt_score["ds"] = bkt_score["ds"].dt.tz_convert(None)

bkt_score = bkt_score.dropna(subset=["ds"])
bkt_score = bkt_score[(bkt_score["ds"] >= EXP_START) & (bkt_score["ds"] <= EXP_END)].reset_index(drop=True)

bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-end"]

bkt_score["partition"] = pd.cut(bkt_score["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
bkt_score = bkt_score.dropna(subset=["partition"]).reset_index(drop=True)

print(bkt_score[["ds","cutoff","partition"]].head(10))
print(bkt_score["partition"].value_counts().sort_index())

          ds     cutoff  partition
0 1990-01-01 1989-12-01  1990-1999
1 1990-02-01 1990-01-01  1990-1999
2 1990-03-01 1990-02-01  1990-1999
3 1990-04-01 1990-03-01  1990-1999
4 1990-05-01 1990-04-01  1990-1999
5 1990-06-01 1990-05-01  1990-1999
6 1990-07-01 1990-06-01  1990-1999
7 1990-08-01 1990-07-01  1990-1999
8 1990-09-01 1990-08-01  1990-1999
9 1990-10-01 1990-09-01  1990-1999
partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-end      68
Name: count, dtype: int64


C:\Users\Mita\AppData\Local\Temp\ipykernel_1756\1537808447.py:5: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):


# Building Leaderbord

In [21]:
import numpy as np
import pandas as pd

tmp = bkt_score.copy()

models = ["LR", "RIDGE", "LGBM"]

# 1) S'assurer que lower <= upper (sécurité)
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"
    tmp[[lo, hi]] = np.sort(tmp[[lo, hi]].to_numpy(), axis=1)

# 2) Wide -> Long + features de scoring
rows = []
base_cols = ["unique_id", "ds", "cutoff", "y", "partition"]

for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"

    s = tmp[base_cols].copy()
    s["model_label"] = m
    s["model_name"]  = m

    s["forecast"] = tmp[m]
    s["lower"]    = tmp[lo]
    s["upper"]    = tmp[hi]

    s["abs_err"]   = (s["y"] - s["forecast"]).abs()
    s["covered"]   = ((s["y"] >= s["lower"]) & (s["y"] <= s["upper"])).astype(int)
    s["int_width"] = (s["upper"] - s["lower"]).abs()

    rows.append(s)

long_sc = pd.concat(rows, ignore_index=True)

# 3) Sanity + types
long_sc = long_sc.loc[:, ~long_sc.columns.duplicated()].copy()
long_sc["partition"] = long_sc["partition"].astype(str)

# 4) Ajouter la partition "ALL" (toutes partitions confondues) SANS DUPLICATION
#    -> on duplique uniquement la colonne "partition" en mettant "ALL"
long_all = long_sc.copy()
long_all["partition"] = "ALL"

long_sc2 = pd.concat([long_sc, long_all], ignore_index=True)

# 5) Score MAE / Coverage / Width / N par modèle et partition (incluant ALL)
score_df = (
    long_sc2
    .groupby(["unique_id", "model_label", "model_name", "partition"], observed=True)
    .agg(
        mae=("abs_err", "mean"),
        coverage=("covered", "mean"),
        width=("int_width", "mean"),
        n=("y", "size"),
    )
    .reset_index()
)

# 6) Top 3 par partition (inclut ALL)
leaderboard = (
    score_df.sort_values(
        by=["partition", "mae", "coverage", "width"],
        ascending=[True, True, False, True],
    )
    .groupby("partition", as_index=False)
    .head(2)
)

print("=== SCORE (MAE / Coverage / Width / N) ===")
print(score_df.sort_values(["partition", "mae"]).head(50))

print("\n=== TOP 2 par partition (incluant ALL) ===")
print(leaderboard)

=== SCORE (MAE / Coverage / Width / N) ===
   unique_id model_label model_name  partition       mae  coverage     width  \
0     UNRATE        LGBM       LGBM  1990-1999  0.434865  0.791667  1.561072   
10    UNRATE       RIDGE      RIDGE  1990-1999  0.483288  0.766667  1.598809   
5     UNRATE          LR         LR  1990-1999  0.508524  0.725000  1.703914   
1     UNRATE        LGBM       LGBM  2000-2008  0.517640  0.629630  1.286702   
6     UNRATE          LR         LR  2000-2008  0.523508  0.629630  1.334418   
11    UNRATE       RIDGE      RIDGE  2000-2008  0.579625  0.657407  1.405270   
12    UNRATE       RIDGE      RIDGE  2009-2019  0.640069  0.810606  2.312753   
7     UNRATE          LR         LR  2009-2019  0.652929  0.810606  2.452591   
2     UNRATE        LGBM       LGBM  2009-2019  0.729249  0.803030  2.404650   
3     UNRATE        LGBM       LGBM   2020-end  1.790125  0.764706  6.739885   
13    UNRATE       RIDGE      RIDGE   2020-end  2.013001  0.750000  6.984155 

# MLFLOW

In [22]:
# =====================================================
# MLflow logging complet (runs = model_label x partition)
# + Dataset name préfixé FEAST_ (ex: stationary_value:value)
# + Tags visibles dans l'UI (model_name, model_label, partition)
# ✅ Log "MLflow Model" (pour que la colonne "Models" ne soit plus "-")
# ❌ PAS de run "model catalog"
# =====================================================

import os
import io
import json
import joblib
import warnings
import logging
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.statsmodels
from contextlib import redirect_stdout, redirect_stderr

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Multivariate_Experimentation")

# =====================================================
# 0) Data log (tous modèles)
# =====================================================
df_log = score_df_all.copy() if "score_df_all" in globals() else score_df.copy()

required_cols = {"model_label", "partition", "mae"}
missing = required_cols - set(df_log.columns)
if missing:
    raise ValueError(f"df_log missing required columns: {missing}")

df_log["model_label"] = df_log["model_label"].astype(str)
df_log["partition"]   = df_log["partition"].astype(str)
if "model_name" in df_log.columns:
    df_log["model_name"] = df_log["model_name"].astype(str)

# =====================================================
# 1) Résumé partitions (dataset input) + FEAST name
# =====================================================
if "bkt_score" not in globals():
    raise ValueError("bkt_score is missing in globals(). Needed for partition_summary_df.")

FEAST_FEATURE_NAME = "stationary_value:value"  # ✅ nom Feast voulu

def _format_feast_dataset_name(feast_name: str) -> str:
    # 'stationary_value:value' -> 'FEAST_stationary_value_value'
    clean = str(feast_name).replace(":", "_").replace("/", "_")
    return f"FEAST_{clean}"

DATASET_NAME = _format_feast_dataset_name(FEAST_FEATURE_NAME)

partition_summary_df = (
    bkt_score
    .groupby("partition", observed=True)
    .agg(
        n_obs=("ds", "size") if "ds" in bkt_score.columns else ("partition", "size"),
        ds_start=("ds", "min") if "ds" in bkt_score.columns else ("partition", "min"),
        ds_end=("ds", "max") if "ds" in bkt_score.columns else ("partition", "max"),
    )
    .reset_index()
)

dataset_obj = mlflow.data.from_pandas(
    partition_summary_df,
    source="feast",
    name=DATASET_NAME
)

# =====================================================
# 2) Helpers
# =====================================================
METRIC_SEQUENCE = "mae>coverage>width"

MODEL_PARAMS = {
    "AR": dict(
        horizon=12,
        min_train_n=36,
        trend="c",
        p_grid=list(range(1, 13)),
        cv_update_every_months=36,
        cv_anchor="1983-01-01",
        use_conformal=True,
        alpha=0.05,
        step_size=12,
        pi_windows=3,
        use_bagging=False
    ),
}

def _get_model_object(model_label: str, partition: str):
    """
    Supporte:
    - models_by_partition[partition] = model_obj
    - models_by_partition[model_label][partition] = model_obj
    """
    if "models_by_partition" not in globals():
        return None, "models_by_partition_missing"

    mbp = globals()["models_by_partition"]

    if isinstance(mbp, dict) and partition in mbp and not isinstance(mbp.get(partition), dict):
        return mbp.get(partition), "precomputed_partition"

    if isinstance(mbp, dict) and model_label in mbp and isinstance(mbp[model_label], dict):
        return mbp[model_label].get(partition), "precomputed_model_partition"

    return None, "not_found"

def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

# =====================================================
# 3) Logging
# =====================================================
os.makedirs("artifacts_tmp", exist_ok=True)

_sink = io.StringIO()
with redirect_stdout(_sink), redirect_stderr(_sink):

    for _, row in df_log.iterrows():

        model_label = str(row["model_label"])
        model_name  = str(row.get("model_name", model_label))
        partition   = str(row["partition"])

        run_name = f"{model_label}_{partition}"

        with mlflow.start_run(run_name=run_name):

            # ✅ Tags visibles dans l'UI (et filtrables)
            mlflow.set_tag("mlflow.runName", run_name)
            mlflow.set_tag("model_label", model_label)
            mlflow.set_tag("model_name", model_name)
            mlflow.set_tag("partition", partition)
            mlflow.set_tag("metric_sequence", METRIC_SEQUENCE)

            # Dataset input (avec nom FEAST_...)
            mlflow.log_input(dataset_obj, context="evaluation")

            # Save dataset summary as artifact
            csv_path = "artifacts_tmp/partition_summary.csv"
            partition_summary_df.to_csv(csv_path, index=False)
            mlflow.log_artifact(csv_path, artifact_path="data_meta")

            # Params
            mlflow.log_param("model_label", model_label)
            mlflow.log_param("model_name", model_name)
            mlflow.log_param("partition", partition)
            mlflow.log_param("metric_sequence", METRIC_SEQUENCE)
            mlflow.log_param("dataset_name", DATASET_NAME)
            mlflow.log_param("feast_feature_name", FEAST_FEATURE_NAME)

            if model_label in MODEL_PARAMS:
                for k, v in MODEL_PARAMS[model_label].items():
                    mlflow.log_param(k, json.dumps(v) if isinstance(v, (list, dict)) else v)

            # Metrics
            mae_val = _safe_float(row.get("mae", None))
            if mae_val is not None:
                mlflow.log_metric("mae", mae_val)

            cov_val = _safe_float(row.get("coverage", None))
            if cov_val is not None and pd.notna(cov_val):
                mlflow.log_metric("coverage", cov_val)

            wid_val = _safe_float(row.get("width", None))
            if wid_val is not None and pd.notna(wid_val):
                mlflow.log_metric("width", wid_val)

            # Leaderboard (artifact)
            leaderboard_path = "artifacts_tmp/leaderboard.csv"
            df_log.to_csv(leaderboard_path, index=False)
            mlflow.log_artifact(leaderboard_path, artifact_path="leaderboard")

            # bkt_score artifacts
            if "bkt_score" in globals():
                os.makedirs("artifacts_tmp", exist_ok=True)

                # Par partition
                if "partition" in bkt_score.columns:
                    df_part = bkt_score[bkt_score["partition"].astype(str) == partition].copy()
                else:
                    df_part = bkt_score.copy()

                if len(df_part) > 0:
                    path_part = f"artifacts_tmp/bkt_{model_label}_{partition}.parquet"
                    df_part.to_parquet(path_part, index=False)
                    mlflow.log_artifact(path_part, artifact_path="data")

                # Full
                path_full = f"artifacts_tmp/bkt_score_full_{model_label}.parquet"
                cols_keep = [c for c in [
                    "ds",
                    "y_true",
                    "y_hat",
                    "lo_95",
                    "hi_95",
                    "partition",
                    "p_selected",
                ] if c in bkt_score.columns]

                if cols_keep:
                    bkt_score[cols_keep].to_parquet(path_full, index=False)
                    mlflow.log_artifact(path_full, artifact_path="data")

            # =====================================================
            # ✅ Model logging MLflow (pour remplir la colonne "Models")
            # =====================================================
            model_obj, model_source = _get_model_object(model_label, partition)

            if model_obj is not None:
                logged_as_mlflow_model = False

                # 1) Essai statsmodels
                try:
                    mlflow.statsmodels.log_model(model_obj, artifact_path="model")
                    logged_as_mlflow_model = True
                    mlflow.set_tag("model_flavor", "statsmodels")
                except Exception:
                    pass

                # 2) Essai sklearn
                if not logged_as_mlflow_model:
                    try:
                        mlflow.sklearn.log_model(model_obj, artifact_path="model")
                        logged_as_mlflow_model = True
                        mlflow.set_tag("model_flavor", "sklearn")
                    except Exception:
                        pass

                # 3) Fallback: artifact joblib si aucun flavor ne marche
                if not logged_as_mlflow_model:
                    model_path = f"artifacts_tmp/{model_label}_model_{partition}.joblib"
                    joblib.dump(model_obj, model_path)
                    mlflow.log_artifact(model_path, artifact_path="model")
                    mlflow.set_tag("model_flavor", "joblib_artifact_only")

                mlflow.log_param("model_logged", True)
                mlflow.log_param("model_source", model_source)
                mlflow.set_tag("model_logged", "true")
                mlflow.set_tag("model_source", model_source)

            else:
                mlflow.log_param("model_logged", False)
                mlflow.log_param("model_source", model_source)
                mlflow.set_tag("model_logged", "false")
                mlflow.set_tag("model_source", model_source)

print("Logging terminé")

Logging terminé
